In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
url = "https://scrapifydatalabs.com/playground/nestly/"

response = requests.get(url, timeout=20)

print("Status code:", response.status_code)
print("Page downloaded:", len(response.text), "characters")

Status code: 200
Page downloaded: 29473 characters


In [3]:
soup = BeautifulSoup(response.text, "html.parser")

cards = soup.select("article.nl-card[data-home-id]")

print("Number of property listings found:", len(cards))

Number of property listings found: 16


In [5]:
properties = []

for card in cards:
    price = card.select_one(".nl-price")
    
    property_data = {
        "Price": price.get_text(strip=True) if price else None,
        "Bedrooms": card.get("data-beds"),
        "Bathrooms": card.get("data-baths"),
        "Area_sqft": card.get("data-sqft")
    }
    
    properties.append(property_data)

properties[:3]

[{'Price': '$625,000', 'Bedrooms': '3', 'Bathrooms': '2', 'Area_sqft': '1680'},
 {'Price': '$419,000', 'Bedrooms': '2', 'Bathrooms': '2', 'Area_sqft': '1045'},
 {'Price': '$875,000', 'Bedrooms': '4', 'Bathrooms': '3', 'Area_sqft': '2450'}]

In [6]:
df = pd.DataFrame(properties)

df

,Price,Bedrooms,Bathrooms,Area_sqft
0,"$625,000",3,2,1680
1,"$419,000",2,2,1045
2,"$875,000",4,3,2450
3,"$512,000",3,2.5,1720
4,"$1,149,000",5,4,3210
5,"$2,450/mo",2,2,980
6,"$389,000",3,2,1410
7,"$299,000",1,1,712
8,"$749,000",4,3,2288
9,"$535,000",3,2,1595


In [8]:
df["Price"] = (
    df["Price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("/mo", "", regex=False)
)

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

df["Bedrooms"] = pd.to_numeric(df["Bedrooms"], errors="coerce")
df["Bathrooms"] = pd.to_numeric(df["Bathrooms"], errors="coerce")
df["Area_sqft"] = pd.to_numeric(df["Area_sqft"], errors="coerce")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Price      16 non-null     int64  
 1   Bedrooms   16 non-null     int64  
 2   Bathrooms  16 non-null     float64
 3   Area_sqft  16 non-null     int64  
dtypes: float64(1), int64(3)
memory usage: 644.0 bytes


In [9]:
print("Number of properties:", len(df))
print("\nMissing values:")
print(df.isnull().sum())

print("\nCleaned data:")
df.head(10)

Number of properties: 16

Missing values:
Price        0
Bedrooms     0
Bathrooms    0
Area_sqft    0
dtype: int64

Cleaned data:


,Price,Bedrooms,Bathrooms,Area_sqft
0,625000,3,2.0,1680
1,419000,2,2.0,1045
2,875000,4,3.0,2450
3,512000,3,2.5,1720
4,1149000,5,4.0,3210
5,2450,2,2.0,980
6,389000,3,2.0,1410
7,299000,1,1.0,712
8,749000,4,3.0,2288
9,535000,3,2.0,1595


In [11]:
print(cards[0].get_text(" | ", strip=True))

House for sale | 🏠 | $625,000 | 3 | bds | | 2 | ba | | 1,680 | sqft | | Single Family | 1842 Maple Ridge Dr, Austin, TX 78745 | 6 days on Nestly | Nestly Premier Realty


In [12]:
addresses = []

for card in cards:
    text = card.get_text(" | ", strip=True)
    
    parts = text.split(" | ")
    
    address = None
    
    for part in parts:
        if "Austin, TX" in part:
            address = part
            break
    
    addresses.append(address)

addresses[:5]

['1842 Maple Ridge Dr, Austin, TX 78745',
 '900 W Cesar Chavez St #412, Austin, TX 78701',
 '4110 Lakewood Cove, Austin, TX 78731',
 '128 Mueller Blvd, Austin, TX 78723',
 '55 Barton Hills Dr, Austin, TX 78704']

In [13]:
df["Address"] = addresses

df

,Price,Bedrooms,Bathrooms,Area_sqft,Address
0,625000,3,2.0,1680,"1842 Maple Ridge Dr, Austin, TX 78745"
1,419000,2,2.0,1045,"900 W Cesar Chavez St #412, Austin, TX 78701"
2,875000,4,3.0,2450,"4110 Lakewood Cove, Austin, TX 78731"
3,512000,3,2.5,1720,"128 Mueller Blvd, Austin, TX 78723"
4,1149000,5,4.0,3210,"55 Barton Hills Dr, Austin, TX 78704"
5,2450,2,2.0,980,"2100 Domain Dr #221, Austin, TX 78758"
6,389000,3,2.0,1410,"902 Delano St, Austin, TX 78721"
7,299000,1,1.0,712,"1800 Barton Springs Rd #7, Austin, TX 78704"
8,749000,4,3.0,2288,"3311 Scenic Brook Dr, Austin, TX 78735"
9,535000,3,2.0,1595,"4702 Duval St, Austin, TX 78751"


In [14]:
df.to_csv("real_estate_listings.csv", index=False)

print("CSV file created successfully!")
print("Rows exported:", len(df))

CSV file created successfully!
Rows exported: 16


In [ ]:
# Verify the exported CSV file

csv_df = pd.read_csv("real_estate_listings.csv")

print("CSV loaded successfully!")
print("Rows:", len(csv_df))
print("Columns:", list(csv_df.columns))

csv_df.head()

CSV loaded successfully!
Rows: 16
Columns: ['Price', 'Bedrooms', 'Bathrooms', 'Area_sqft', 'Address']


,Price,Bedrooms,Bathrooms,Area_sqft,Address
0,625000,3,2.0,1680,"1842 Maple Ridge Dr, Austin, TX 78745"
1,419000,2,2.0,1045,"900 W Cesar Chavez St #412, Austin, TX 78701"
2,875000,4,3.0,2450,"4110 Lakewood Cove, Austin, TX 78731"
3,512000,3,2.5,1720,"128 Mueller Blvd, Austin, TX 78723"
4,1149000,5,4.0,3210,"55 Barton Hills Dr, Austin, TX 78704"


: 